# AAI-510 Final Project — Notebook 1: Data Pipeline
**Owner:** Idrees Khan (Data Engineer)
**Team:** Group 7 — Wael Alhathal & Idrees Khan
**Project:** Nova — Order & Returns Support Assistant for Nestwell

---

## What this notebook does

This notebook is the full data engineering pipeline for our project. It covers:

1. Loading our synthetic Olist-style dataset (based on the real Olist Brazilian E-commerce dataset from Kaggle)
2. Cleaning and transforming it into a simplified orders table
3. Registering it in Unity Catalog so the agent tools can query it
4. Defining and registering `order_lookup` as a Unity Catalog SQL function
5. Defining and registering `policy_lookup` as a Unity Catalog Python function
6. Running quick unit tests on both tools to make sure they work before the agent uses them

### Why we built it this way

The raw Olist dataset uses Portuguese product names and category codes that are hard
to present in a customer-facing response. Per our status update, we scoped the agent
to order-ID lookups only, which means the two tools cover everything the agent needs:
one structured lookup over real order data, and one text search over our short policy doc.

## 0. Setup

In [0]:
# Install any missing packages (Databricks usually has these already)
# %pip install pandas --quiet

import pandas as pd
import os
from datetime import datetime

# Unity Catalog settings — update these to match your workspace
CATALOG   = "main"
SCHEMA    = "nestwell"
TABLE     = "orders"
FULL_PATH = f"{CATALOG}.{SCHEMA}.{TABLE}"

print(f"Target table: {FULL_PATH}")
print(f"Timestamp   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Target table: main.nestwell.orders
Timestamp   : 2026-06-06 19:18:22


## 1. Load Raw CSVs

We generated three CSVs from our synthetic data script (modeled on the Olist schema):
- `nestwell_orders.csv` — one row per order with status, timestamps, return eligibility
- `nestwell_order_items.csv` — line items per order
- `nestwell_customers.csv` — anonymized customer records

In Databricks, upload these CSVs to your workspace file store or DBFS and update the
paths below. For local testing, relative paths work fine.

In [0]:
# ── Adjust these paths to wherever you uploaded the CSVs in your workspace ──
BASE = "/Volumes/main/nestwell/data"

try:
    df_orders    = pd.read_csv(f"{BASE}/nestwell_orders.csv")
    df_items     = pd.read_csv(f"{BASE}/nestwell_order_items.csv")
    df_customers = pd.read_csv(f"{BASE}/nestwell_customers.csv")
    print("Loaded from Workspace path")
except Exception:
    # Fallback: load locally if running outside Databricks
    df_orders    = pd.read_csv("nestwell_orders.csv")
    df_items     = pd.read_csv("nestwell_order_items.csv")
    df_customers = pd.read_csv("nestwell_customers.csv")
    print("Loaded from local path (non-Databricks fallback)")

print(f"\nOrders    : {len(df_orders):,} rows")
print(f"Items     : {len(df_items):,} rows")
print(f"Customers : {len(df_customers):,} rows")

Loaded from Workspace path

Orders    : 800 rows
Items     : 1,207 rows
Customers : 300 rows


## 2. Inspect the Raw Data

In [0]:
print("=== Orders sample ===")
display(df_orders.head(5))

=== Orders sample ===


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,return_eligible,days_since_purchase
ORD-1000,01CA2F7A,delivered,2023-06-22 12:13:55,2023-06-22 16:13:55,2023-06-23 16:13:55,2023-06-29 16:13:55,2023-06-29,false,192
ORD-1001,F038F576,shipped,2023-11-11 00:55:14,2023-11-11 02:55:14,2023-11-13 02:55:14,null,2023-11-23,false,50
ORD-1002,04DED5CA,delivered,2023-10-26 19:41:13,2023-10-26 23:41:13,2023-10-29 23:41:13,2023-11-04 23:41:13,2023-11-07,false,66
ORD-1003,5711A536,delivered,2023-11-14 05:58:41,2023-11-14 07:58:41,2023-11-16 07:58:41,2023-11-18 07:58:41,2023-11-26,false,47
ORD-1004,E668A4FB,canceled,2023-08-04 08:41:39,null,null,null,null,false,149


In [0]:
print("=== Status distribution ===")
print(df_orders["order_status"].value_counts().to_string())

=== Status distribution ===
delivered     497
shipped       167
processing     60
canceled       49
invoiced       27


In [0]:
print("=== Return-eligible orders ===")
print(f"  Total orders        : {len(df_orders)}")
print(f"  Return eligible     : {df_orders['return_eligible'].sum()}")
print(f"  Not eligible        : {(~df_orders['return_eligible']).sum()}")

=== Return-eligible orders ===
  Total orders        : 800
  Return eligible     : 35
  Not eligible        : 765


In [0]:
print("=== Missing value check ===")
missing = df_orders.isnull().sum()
print(missing[missing > 0].to_string() if missing.any() else "No nulls found.")

=== Missing value check ===
order_approved_at                 49
order_delivered_carrier_date     136
order_delivered_customer_date    303
order_estimated_delivery_date     49


## 3. Clean & Transform

We do a few light transformations:
- Join customer name onto orders so `order_lookup` can greet by name
- Compute `item_count` and `order_total` from the items table
- Parse timestamps and standardize date formats
- Fill empty strings (canceled orders have no delivery date) with None

In [0]:
# 3a. Aggregate items per order
agg_items = (
    df_items
    .groupby("order_id")
    .agg(
        item_count   = ("order_item_id", "count"),
        order_total  = ("price", "sum"),
    )
    .reset_index()
)
agg_items["order_total"] = agg_items["order_total"].round(2)

# 3b. Join customer name
df_merged = (
    df_orders
    .merge(df_customers[["customer_id", "customer_name"]], on="customer_id", how="left")
    .merge(agg_items, on="order_id", how="left")
)

# 3c. Normalize empty strings → None (better for SQL)
str_cols = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in str_cols:
    df_merged[col] = df_merged[col].replace("", None)

# 3d. Select and rename final columns
df_clean = df_merged[[
    "order_id",
    "customer_id",
    "customer_name",
    "order_status",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
    "return_eligible",
    "days_since_purchase",
    "item_count",
    "order_total",
]].copy()

print(f"Clean table shape: {df_clean.shape}")
display(df_clean.head(5))

Clean table shape: (800, 11)


order_id,customer_id,customer_name,order_status,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,return_eligible,days_since_purchase,item_count,order_total
ORD-1000,01CA2F7A,Sarah Garcia,delivered,2023-06-22 12:13:55,2023-06-29,2023-06-29 16:13:55,false,192,1,77.84
ORD-1001,F038F576,Michael Nguyen,shipped,2023-11-11 00:55:14,2023-11-23,null,false,50,2,223.57
ORD-1002,04DED5CA,Maria Martinez,delivered,2023-10-26 19:41:13,2023-11-07,2023-11-04 23:41:13,false,66,1,128.85
ORD-1003,5711A536,Sarah Taylor,delivered,2023-11-14 05:58:41,2023-11-26,2023-11-18 07:58:41,false,47,3,426.96
ORD-1004,E668A4FB,Thomas Williams,canceled,2023-08-04 08:41:39,null,null,false,149,2,256.74


## 4. Write to Unity Catalog

We create the schema if it doesn't exist, then write the clean DataFrame as a
Delta table in Unity Catalog. This is what `order_lookup` queries.

In [0]:
# Create schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Schema ready: {CATALOG}.{SCHEMA}")

Schema ready: main.nestwell


In [0]:
# Convert pandas → Spark and write as Delta table
spark_df = spark.createDataFrame(df_clean)

(
    spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_PATH)
)

print(f"Table written: {FULL_PATH}")
print(f"Row count    : {spark.table(FULL_PATH).count():,}")

Table written: main.nestwell.orders
Row count    : 800


In [0]:
# Quick sanity check — query the table directly
display(spark.sql(f"SELECT * FROM {FULL_PATH} LIMIT 5"))

order_id,customer_id,customer_name,order_status,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,return_eligible,days_since_purchase,item_count,order_total
ORD-1000,01CA2F7A,Sarah Garcia,delivered,2023-06-22 12:13:55,2023-06-29,2023-06-29 16:13:55,false,192,1,77.84
ORD-1001,F038F576,Michael Nguyen,shipped,2023-11-11 00:55:14,2023-11-23,null,false,50,2,223.57
ORD-1002,04DED5CA,Maria Martinez,delivered,2023-10-26 19:41:13,2023-11-07,2023-11-04 23:41:13,false,66,1,128.85
ORD-1003,5711A536,Sarah Taylor,delivered,2023-11-14 05:58:41,2023-11-26,2023-11-18 07:58:41,false,47,3,426.96
ORD-1004,E668A4FB,Thomas Williams,canceled,2023-08-04 08:41:39,null,null,false,149,2,256.74


## 5. Register `order_lookup` as a Unity Catalog SQL Function

This is the structured tool the agent calls to look up an order by its ID.
It returns everything the agent needs to answer "where is my order" questions.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.order_lookup(order_id_input STRING)
RETURNS TABLE (
    order_id                      STRING,
    customer_name                 STRING,
    order_status                  STRING,
    order_purchase_timestamp      STRING,
    order_estimated_delivery_date STRING,
    order_delivered_customer_date STRING,
    return_eligible               BOOLEAN,
    days_since_purchase           INT,
    item_count                    INT,
    order_total                   DOUBLE
)
COMMENT 'Looks up a Nestwell order by order ID and returns status, delivery dates, and return eligibility.'
RETURN
    SELECT
        order_id,
        customer_name,
        order_status,
        order_purchase_timestamp,
        order_estimated_delivery_date,
        order_delivered_customer_date,
        return_eligible,
        days_since_purchase,
        item_count,
        order_total
    FROM {FULL_PATH}
    WHERE order_id = order_id_input
""")

print(f"Function registered: {CATALOG}.{SCHEMA}.order_lookup")

Function registered: main.nestwell.order_lookup


## 6. Register `policy_lookup` as a Unity Catalog Python Function

This tool reads the Nestwell policy document and returns the section most relevant
to the customer's question. Because the policy is only ~500 words, we do simple
keyword matching — no vector search needed, and the whole section fits in context.

In [0]:
# First, read and register the policy content so the function can access it.
# In a real Databricks environment, you would store this in a volume or config table.
# For this project we inline it as a constant inside the function body.

with open(f"{BASE}/nestwell_policy.txt", "r") as f:
    policy_text = f.read()

print(f"Policy doc loaded — {len(policy_text)} characters")

Policy doc loaded — 4197 characters


In [0]:
# We embed the policy directly in the function so no external file access is needed.
# The function splits the doc into sections and returns the best match.

policy_escaped = policy_text.replace("'", "\\'")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.policy_lookup(topic STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Returns the Nestwell policy section most relevant to the given topic keyword.'
AS $$
POLICY_DOC = '''{policy_escaped}'''

topic_lower = topic.lower()

keyword_map = {{
    "return":    ["return eligibility", "how to start a return"],
    "refund":    ["refund timelines"],
    "shipping":  ["shipping options", "shipping costs"],
    "damage":    ["damaged or incorrect"],
    "exchange":  ["exchanges"],
    "contact":   ["contact", "escalation"],
}}

sections = [s.strip() for s in POLICY_DOC.split("━" * 10) if s.strip()]

# Try keyword map first
for kw, section_hints in keyword_map.items():
    if kw in topic_lower:
        for sec in sections:
            for hint in section_hints:
                if hint.lower() in sec.lower():
                    return sec

# Fallback: return any section containing the topic word
for sec in sections:
    if topic_lower in sec.lower():
        return sec

# Last resort: return full policy
return POLICY_DOC
$$
""")

print(f"Function registered: {CATALOG}.{SCHEMA}.policy_lookup")

Function registered: main.nestwell.policy_lookup


## 7. Unit Tests

Before handing the tools off to the agent, we run a quick test on each one
to make sure they return sensible output.

### 7a. Test order_lookup

In [0]:
# Pick a real order ID from our data to test
test_order_id = df_clean["order_id"].iloc[0]
print(f"Testing with order_id: {test_order_id}\n")

result = spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.order_lookup('{test_order_id}')
""")
display(result)

Testing with order_id: ORD-1000



order_id,customer_name,order_status,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,return_eligible,days_since_purchase,item_count,order_total
ORD-1000,Sarah Garcia,delivered,2023-06-22 12:13:55,2023-06-29,2023-06-29 16:13:55,false,192,1,77.84


In [0]:
# Test with a non-existent order — should return 0 rows, not an error
result_empty = spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.order_lookup('ORD-FAKE-999')
""")
print(f"Rows returned for fake order: {result_empty.count()} (expected 0)")

Rows returned for fake order: 0 (expected 0)


In [0]:
# Test a few different statuses
for status in ["delivered", "shipped", "canceled"]:
    sample = df_clean[df_clean["order_status"] == status]["order_id"].iloc[0]
    row = spark.sql(f"SELECT order_id, order_status, return_eligible FROM {CATALOG}.{SCHEMA}.order_lookup('{sample}')").collect()
    print(f"Status={status:12s} | order_id={sample} | return_eligible={row[0]['return_eligible']}")

Status=delivered    | order_id=ORD-1000 | return_eligible=False
Status=shipped      | order_id=ORD-1001 | return_eligible=False
Status=canceled     | order_id=ORD-1004 | return_eligible=False


### 7b. Test policy_lookup

In [0]:
test_topics = ["return", "refund", "shipping", "damage", "exchange"]

for topic in test_topics:
    result = spark.sql(f"SELECT {CATALOG}.{SCHEMA}.policy_lookup('{topic}') AS section")
    section_text = result.collect()[0]["section"]
    # Just show the first 200 chars so output stays readable
    preview = section_text[:200].replace("\n", " ").strip()
    print(f"\nTopic: '{topic}'")
    print(f"  → {preview}...")


Topic: 'return'
  → SECTION 1 — RETURN ELIGIBILITY          Most items sold by Nestwell can be returned within 30 days of the delivery date,     provided the item is unused, in its original packaging, and accompanied by...

Topic: 'refund'
  → SECTION 3 — REFUND TIMELINES          Refunds are issued to the original payment method only.            • Credit / Debit Card: 5–7 business days after the return is processed       • PayPal: 3–5 busi...

Topic: 'shipping'
  → SECTION 4 — SHIPPING OPTIONS & COSTS          Standard Shipping (5–8 business days)       • Free on orders over $75       • $5.99 on orders under $75          Expedited Shipping (2–3 business days)...

Topic: 'damage'
  → SECTION 5 — DAMAGED OR INCORRECT ITEMS          If your item arrived damaged or you received the wrong item, please contact us     within 7 days of delivery. We will arrange a free return and ship a r...

Topic: 'exchange'
  → SECTION 6 — EXCHANGES          Nestwell does not offer direct exchanges at this

### 7c. Edge case — topic not in policy

In [0]:
result = spark.sql(f"SELECT {CATALOG}.{SCHEMA}.policy_lookup('warranty') AS section")
section_text = result.collect()[0]["section"]
print(f"Topic: 'warranty'")
print(f"Chars returned: {len(section_text)}")
print("(Falls back to full policy doc — agent will say it can't find specific info)")

Topic: 'warranty'
Chars returned: 4609
(Falls back to full policy doc — agent will say it can't find specific info)


## 8. Summary

| Artifact | Location |
|---|---|
| Clean orders table | `main.nestwell.orders` |
| order_lookup function | `main.nestwell.order_lookup` |
| policy_lookup function | `main.nestwell.policy_lookup` |

Both tools are registered, tested, and ready for the agent in Notebook 2.

**Note on the Portuguese names issue (per our status update):** The raw Olist dataset
uses Portuguese product category codes like `cama_mesa_banho` and numeric product IDs.
We replaced these with English category names and readable product names during data
generation, and we scoped the agent to order-ID lookups so it never has to surface
raw product codes in a customer response. This keeps every answer grounded in
something the tools can actually return cleanly.